In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

env: ANYWIDGET_HMR=1


In [2]:
import celldega as dega
import numpy as np
import pandas as pd
from anndata import AnnData
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import scanpy as sc
from spatialdata_io import xenium
import spatialdata as sd
import os
from scipy.sparse import csr_matrix
print(dega.__version__)

objc[72274]: Class GNotificationCenterDelegate is implemented in both /opt/homebrew/Cellar/glib/2.84.3/lib/libgio-2.0.0.dylib (0x1778c84b8) and /Users/jishar/anaconda3/lib/libgio-2.0.0.dylib (0x17747e310). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


0.13.0a9


In [3]:
from ipywidgets import Widget
Widget.close_all()

## Real Data

In [4]:
data_dir = "data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs"

## Make AnnData

In [5]:
# # Ingest xenium data raw output folder using spatialdata-io
# sdata = xenium(data_dir)

# # Write sdata to a zarr file
# zarr_path = f"{data_dir}.zarr"

# # Check if the zarr file already exists
# if os.path.exists(zarr_path):
#     print(f"The file {zarr_path} already exists.")
# else:
#     # If the file does not exist, write the data
#     sdata.write(zarr_path)
#     print(f"Data written to {zarr_path} successfully.")

# # Read zarr file using spatialdata
# sdata = sd.read_zarr(zarr_path)

# # Create anndata from sdata.tables['table]
# adata = sdata.tables["table"]
# adata.write_h5ad(f'{data_dir}.h5ad')

In [6]:
# # Load h5ad file
# adata = sc.read_h5ad(f'{data_dir}.h5ad')
# adata.obs.set_index('cell_id', inplace=True)
# adata

## Scanpy processing

In [7]:
# sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)
# sc.pp.filter_cells(adata, min_counts=10)
# sc.pp.filter_genes(adata, min_cells=5)

# adata.X = csr_matrix(adata.X)

# adata.layers["counts"] = adata.X

# sc.pp.normalize_total(adata, inplace=True)

# sc.pp.log1p(adata)

# sc.pp.highly_variable_genes(adata, n_top_genes=2000)
# adata = adata[:, adata.var.highly_variable].copy()

# sc.pp.pca(adata)

# sc.pp.neighbors(adata, use_rep='X_pca', n_neighbors=10)

# sc.tl.leiden(adata, resolution=1.0)

In [8]:
# adata.write_h5ad(f'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [9]:
adata = sc.read_h5ad(f'data/xenium_data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_sc_processed.h5ad')

In [10]:
# Convert the results to a pandas DataFrame
def get_ranked_genes_df(adata, n_genes=100):
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    dfs = []
    for group in groups:
        df = pd.DataFrame({
            'gene': result['names'][group][:n_genes],
            'logfoldchanges': result['logfoldchanges'][group][:n_genes],
            'pvals': result['pvals'][group][:n_genes],
            'pvals_adj': result['pvals_adj'][group][:n_genes],
            'scores': result['scores'][group][:n_genes],
            'cluster': group
        })
        dfs.append(df)
    return pd.concat(dfs)

## Rank and save marker genes

In [11]:
# # Run ranking (faster)
# sc.tl.rank_genes_groups(adata, groupby="leiden", method="t-test", use_raw=False, show_progress=True)

# # Save markers
# marker_df = get_ranked_genes_df(adata, n_genes=100)
# marker_df.to_csv("data/xenium_data/marker_genes_by_cluster.csv", index=False)

#### 1. Uploaded "marker_genes_by_cluster.csv" on ChatGPT, and asked for tentative cell types.
#### 2. "Predicted_Cell_Types_by_Cluster.csv" has the predicted cell types for each cluster based on the top 10 marker genes, with the cluster number included in the label.

In [12]:
pred_cell_types_df = pd.read_csv("data/xenium_data/Predicted_Cell_Types_by_Cluster.csv")
pred_cell_types_df.drop(['Unnamed: 0'], axis=1, inplace=True)
pred_cell_types_df["category"] = pred_cell_types_df["predicted_cell_type"].str.split("_").str[0]
pred_cell_types_df

,cluster,predicted_cell_type,category
0,0,Unknown_0,Unknown
1,1,Epithelial_1,Epithelial
2,2,Smooth muscle_2,Smooth muscle
3,3,Fibroblast_3,Fibroblast
4,4,Epithelial_4,Epithelial
5,5,Unknown_5,Unknown
6,6,Macrophage_6,Macrophage
7,7,Endothelial_7,Endothelial
8,8,T cell_8,T cell
9,9,Endothelial_9,Endothelial


### Make hextiles

In [60]:
data = dega.nbhd._get_gdf_cell(adata)
gdf_nbhd = dega.nbhd.generate_hex_grid(data, diameter=50)

In [61]:
adata_nbp, gdf_nbhd = dega.nbhd.calc_nbp(data, gdf_nbhd, category="cluster")

Calculating NBP


In [62]:
adata_nbp.obs

""
name
hex_1000
hex_10000
hex_10001
hex_10002
hex_10003
...
hex_9995
hex_9996
hex_9997


## Cell-cluster by Hextile using NBHD module methods

In [53]:
# Clustering
sc.pp.normalize_total(adata_nbp, inplace=True)
sc.pp.log1p(adata_nbp)
sc.pp.neighbors(adata_nbp, n_neighbors=10)
sc.tl.leiden(adata_nbp, resolution=1)

In [54]:
population_distribution = pd.DataFrame(
    adata_nbp.X, index=adata_nbp.obs_names, columns=adata_nbp.var_names
)

In [55]:
# Add clustering and proportions to hex GeoDataFrame
gdf_nbhd = gdf_nbhd.set_index("name")
gdf_nbhd["leiden"] = adata_nbp.obs["leiden"].values
gdf_nbhd["niche"] = [f"niche_{cluster}" for cluster in adata_nbp.obs["leiden"].values]
gdf_nbhd = gdf_nbhd.join(population_distribution)
gdf_nbhd.reset_index(inplace=True)

In [56]:
# Dissolve to form niches
gdf_niche = dega.nbhd._dissolve_by_category(gdf_nbhd, "leiden")
gdf_niche["name"] = [f"niche_{c}" for c in gdf_niche["leiden"]]

In [57]:
gdf_nbhd

,name,geometry,leiden,niche,0,1,2,3,4,5,...,21,22,23,24,25,26,27,28,29,30
0,hex_24,"POLYGON ((6205.05247 33.89656, 6183.40184 46.3...",0,niche_0,0.0,0.000000,0.0,0.154151,0.559616,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,hex_25,"POLYGON ((6248.35374 33.89656, 6226.70311 46.3...",61,niche_61,0.0,0.182322,0.0,0.470004,0.000000,0.182322,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,hex_26,"POLYGON ((6291.65501 33.89656, 6270.00438 46.3...",11,niche_11,0.0,0.154151,0.0,0.223144,0.459532,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,hex_27,"POLYGON ((6334.95628 33.89656, 6313.30565 46.3...",18,niche_18,0.0,0.154151,0.0,0.000000,0.559616,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,hex_28,"POLYGON ((6378.25755 33.89656, 6356.60692 46.3...",35,niche_35,0.0,0.405465,0.0,0.154151,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28816,hex_51212,"POLYGON ((7374.18677 7983.89656, 7352.53613 79...",3,niche_3,0.0,0.000000,0.0,0.336472,0.336472,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28817,hex_51213,"POLYGON ((7417.48804 7983.89656, 7395.83740 79...",3,niche_3,0.0,0.000000,0.0,0.117783,0.223144,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28818,hex_51214,"POLYGON ((7460.78931 7983.89656, 7439.13867 79...",3,niche_3,0.0,0.000000,0.0,0.223144,0.117783,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28819,hex_51215,"POLYGON ((7504.09058 7983.89656, 7482.43994 79...",67,niche_67,0.0,0.000000,0.0,0.080043,0.154151,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## SKIP - Clustergram: hextile_nbhd-by-cell_population

In [19]:
# gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
# gdf_nbhd_.set_index('name', inplace=True)
# gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
# gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
# gdf_nbhd_.head()

In [20]:
# meta_col = pd.DataFrame(index=gdf_nbhd_.columns.tolist())
# top_cols = [int(col) for col in gdf_nbhd_.sum(axis=0).sort_values(ascending=False).index]
# predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
# meta_col['category'] = predicted_types
# meta_col[:5]

In [21]:
# meta_row = pd.DataFrame(index=gdf_nbhd_.index.tolist())
# top_rows = gdf_nbhd_.sum(axis=1).sort_values(ascending=False).index.tolist()
# niches = gdf_nbhd.set_index("name").loc[top_rows, "niche"].tolist()
# meta_row['niche'] = niches
# meta_row[:5]

In [22]:
# mat = dega.clust.Matrix(
#     gdf_nbhd_,
#     name='parquet',
#     meta_col=meta_col,
#     col_attr=['category'],
#     meta_row=meta_row
# )

# mat.norm(axis='row', by='zscore')
# mat.clust()
# cgm = dega.viz.Clustergram(
#     matrix=mat, 
#     width=500, 
#     height=500
# )
# cgm

## Clustergram: cell_population-by-hextile_nbhd 

In [23]:
gdf_nbhd_ = gdf_nbhd.drop(['geometry', 'leiden', 'niche'], axis=1)
gdf_nbhd_.set_index('name', inplace=True)
gdf_nbhd_ = gdf_nbhd_[(gdf_nbhd_ != 0).any(axis=1)]
gdf_nbhd_ = gdf_nbhd_.loc[gdf_nbhd_.std(axis=1) != 0]
gdf_nbhd_.head()

,0,1,2,3,4,5,6,7,8,9,...,21,22,23,24,25,26,27,28,29,30
name,,,,,,,,,,,,,,,,,,,,,
hex_24,0.0,0.000000,0.0,0.154151,0.559616,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_25,0.0,0.182322,0.0,0.470004,0.000000,0.182322,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_26,0.0,0.154151,0.0,0.223144,0.459532,0.000000,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_27,0.0,0.154151,0.0,0.000000,0.559616,0.000000,0.080043,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
hex_28,0.0,0.405465,0.0,0.154151,0.000000,0.000000,0.154151,0.0,0.154151,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
# Transpose neighborhood matrix
gdf_nbhd_T = gdf_nbhd_.T

# Rename index and column names
gdf_nbhd_T.index.name = "cluster"
gdf_nbhd_T.columns.name = ""

# Ensure cluster index is integer
gdf_nbhd_T.index = gdf_nbhd_T.index.astype(int)

# Create cluster → cell type mapping
cluster_to_cell_type = pred_cell_types_df.set_index("cluster")["predicted_cell_type"].to_dict()

# Map cluster index to predicted cell type
gdf_nbhd_T.index = gdf_nbhd_T.index.map(cluster_to_cell_type)

# Check for unmapped values
if gdf_nbhd_T.index.isnull().any():
    raise ValueError("Some clusters could not be mapped to predicted cell types.")

# Build metadata for rows
top_rows = gdf_nbhd_T.sum(axis=1).sort_values(ascending=False).index
predicted_types_df = pred_cell_types_df.set_index("predicted_cell_type")

# Filter and align with top rows (cell types)
predicted_types = predicted_types_df.loc[top_rows, "category"]
meta_row = pd.DataFrame(index=top_rows)
meta_row['category'] = predicted_types.values

# Build metadata for columns
top_cols = gdf_nbhd_T.sum(axis=0).sort_values(ascending=False).index
niches_df = gdf_nbhd.set_index("name")

# Filter and align with top columns (neighborhoods)
niches = niches_df.loc[top_cols, "niche"]
meta_col = pd.DataFrame(index=top_cols)
meta_col['niche'] = niches.values

In [25]:
mat = dega.clust.Matrix(
    gdf_nbhd_T,
    name='parquet',
    meta_col=meta_col,
    col_attr=['niche'],
    meta_row=meta_row
)

# mat.downsample_to(axis='col', category='niche')
mat.norm(axis='row', by='zscore')
mat.clust()
cgm = dega.viz.Clustergram(
    matrix=mat, 
    width=500, 
    height=500
)
cgm

Clustergram(height=500, network_meta={'linkage': {}, 'row_attr': ['category'], 'col_attr': ['niche'], 'row_att…

## Visualize in Landscape view: Hextile NBHD

In [46]:
gdf_nbhd

,name,geometry,leiden,niche,0,1,2,3,4,5,...,21,22,23,24,25,26,27,28,29,30
0,hex_24,"POLYGON ((6205.05247 33.89656, 6183.40184 46.3...",0,niche_0,0.0,0.000000,0.0,0.154151,0.559616,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,hex_25,"POLYGON ((6248.35374 33.89656, 6226.70311 46.3...",61,niche_61,0.0,0.182322,0.0,0.470004,0.000000,0.182322,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,hex_26,"POLYGON ((6291.65501 33.89656, 6270.00438 46.3...",11,niche_11,0.0,0.154151,0.0,0.223144,0.459532,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,hex_27,"POLYGON ((6334.95628 33.89656, 6313.30565 46.3...",18,niche_18,0.0,0.154151,0.0,0.000000,0.559616,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,hex_28,"POLYGON ((6378.25755 33.89656, 6356.60692 46.3...",35,niche_35,0.0,0.405465,0.0,0.154151,0.000000,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28816,hex_51212,"POLYGON ((7374.18677 7983.89656, 7352.53613 79...",3,niche_3,0.0,0.000000,0.0,0.336472,0.336472,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28817,hex_51213,"POLYGON ((7417.48804 7983.89656, 7395.83740 79...",3,niche_3,0.0,0.000000,0.0,0.117783,0.223144,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28818,hex_51214,"POLYGON ((7460.78931 7983.89656, 7439.13867 79...",3,niche_3,0.0,0.000000,0.0,0.223144,0.117783,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28819,hex_51215,"POLYGON ((7504.09058 7983.89656, 7482.43994 79...",67,niche_67,0.0,0.000000,0.0,0.080043,0.154151,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
gdf_nbhd_LF = gdf_nbhd.copy()
gdf_nbhd_LF = gdf_nbhd_LF[['geometry','name','leiden']]
gdf_nbhd_LF.rename(columns={'leiden':'cat'}, inplace=True)
gdf_nbhd_LF.head()

,geometry,name,cat
0,"POLYGON ((6205.05247 33.89656, 6183.40184 46.3...",hex_24,0
1,"POLYGON ((6248.35374 33.89656, 6226.70311 46.3...",hex_25,61
2,"POLYGON ((6291.65501 33.89656, 6270.00438 46.3...",hex_26,11
3,"POLYGON ((6334.95628 33.89656, 6313.30565 46.3...",hex_27,18
4,"POLYGON ((6378.25755 33.89656, 6356.60692 46.3...",hex_28,35


In [45]:
# Step 1: Create the mapping from `cat` number to cell type string
# Assuming pred_cell_types_df has: 'cluster' (int), 'predicted_cell_type' (str)

cat_to_cell_type = pred_cell_types_df.set_index("cluster")["predicted_cell_type"].to_dict()

# Step 2: Apply the mapping to `gdf_nbhd_LF`
gdf_nbhd_LF = gdf_nbhd_LF.copy()  # to avoid modifying original in-place
gdf_nbhd_LF["predicted_cell_type"] = gdf_nbhd_LF["cat"].map(cat_to_cell_type)

# Optional safety check
if gdf_nbhd_LF["predicted_cell_type"].isnull().any():
    unmapped = gdf_nbhd_LF[gdf_nbhd_LF["predicted_cell_type"].isnull()]["cat"].unique()
    print(f"⚠️ Warning: Some 'cat' values were not mapped: {unmapped}")


⚠️ Warning: Some 'cat' values were not mapped: ['0', '61', '11', '18', '35', ..., '101', '102', '136', '125', '148']
Length: 155
Categories (155, object): ['0', '1', '2', '3', ..., '151', '152', '153', '154']


In [27]:
categories = gdf_nbhd_LF['cat'].cat.categories
n_cats = len(categories)

cmap = matplotlib.colormaps['tab20']
colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

cat_to_hex = dict(zip(categories, colors))

gdf_nbhd_LF['color'] = gdf_nbhd_LF['cat'].astype(str).map(cat_to_hex)
gdf_nbhd_LF.head()

,geometry,name,cat,color
0,"POLYGON ((6205.05247 33.89656, 6183.40184 46.3...",hex_24,0,#1f77b4
1,"POLYGON ((6248.35374 33.89656, 6226.70311 46.3...",hex_25,61,#ff9896
2,"POLYGON ((6291.65501 33.89656, 6270.00438 46.3...",hex_26,11,#aec7e8
3,"POLYGON ((6334.95628 33.89656, 6313.30565 46.3...",hex_27,18,#ff7f0e
4,"POLYGON ((6378.25755 33.89656, 6356.60692 46.3...",hex_28,35,#2ca02c


In [37]:
# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
# data_dir = f'data/xenium_data/'
# path_landscape_files=f'data/landscape_files/{sample}_test'
# base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

base_url = 'https://raw.githubusercontent.com/broadinstitute/celldega_Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs_v2/main/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'

landscape_ist = dega.viz.Landscape(
    technology="Xenium",
    base_url = base_url,
    nbhd=gdf_nbhd_LF
)

# landscape_ist

In [38]:
dega.viz.landscape_clustergram(landscape_ist, cgm)

## SKIP - Visualize in Landscape view: Niche NBHD

In [30]:
# gdf_niche_LF = gdf_niche.copy()
# gdf_niche_LF = gdf_niche_LF[['geometry','name','leiden']]
# gdf_niche_LF.rename(columns={'leiden':'cat'}, inplace=True)

# # Convert 'cat' from categorical strings like '0' → int → +1 → str again
# # gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(int) + 1
# # gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype(str)
# # gdf_niche_LF['cat'] = gdf_niche_LF['cat'].astype('category')

# gdf_niche_LF.head()

In [31]:
# categories = gdf_niche_LF['cat'].cat.categories
# n_cats = len(categories)

# cmap = matplotlib.colormaps['tab20']
# colors = [matplotlib.colors.to_hex(cmap(i / n_cats)) for i in range(n_cats)]

# cat_to_hex = dict(zip(categories, colors))

# gdf_niche_LF['color'] = gdf_niche_LF['cat'].astype(str).map(cat_to_hex)
# gdf_niche_LF.head()

In [32]:
# sample = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
# data_dir = f'data/xenium_data/'
# path_landscape_files=f'data/landscape_files/{sample}_test'

# landscape_ist = dega.viz.Landscape(
#     technology="Xenium",
#     base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
#     nbhd=gdf_niche_LF
# )

# landscape_ist

## SKIP - Clustergram: niche_nbhd-by-cell_population

In [33]:
# gdf_niche_ = gdf_niche.drop(['geometry', 'leiden'], axis=1)
# gdf_niche_.set_index('name', inplace=True)

# gdf_niche_ = gdf_niche_.apply(pd.to_numeric, errors='coerce')
# gdf_niche_ = gdf_niche_.replace([np.inf, -np.inf], np.nan).fillna(0)
# gdf_niche_ = gdf_niche_[(gdf_niche_ != 0).any(axis=1)]
# gdf_niche_ = gdf_niche_.loc[gdf_niche_.std(axis=1) != 0]
# gdf_niche_ = gdf_niche_.loc[:, gdf_niche_.std(axis=0) != 0]

# assert np.isfinite(gdf_niche_.values).all(), "Matrix still contains non-finite values!"

In [34]:
# meta_col = pd.DataFrame(index=gdf_niche_.columns.tolist())
# top_cols = [int(col) for col in gdf_niche_.sum(axis=0).sort_values(ascending=False).index]
# predicted_types = pred_cell_types_df.set_index("cluster").loc[top_cols, "category"].tolist()
# meta_col['category'] = predicted_types
# meta_col[:5]

In [35]:
# mat = dega.clust.Matrix(
#     gdf_niche_,
#     name='parquet',
#     meta_col=meta_col,
#     col_attr=['category'],
# )

# mat.clust()
# cgm = dega.viz.Clustergram(
#     matrix=mat, 
#     width=500, 
#     height=500
# )
# cgm